# P21 review round 5 controls
nfdtr: lessons written with the TRAIN-ONLY data profile and no evaluator outcome (P21 writer --train-only). C1-nfdtr / E-nfdtr on that store. A0g2 / C1g2: P12's no-memory and all-own-lessons arms rerun on the 2-GPU servers, to check the hardware split. Predictions: `quick_runs/predictions/P21.md`.

In [1]:
import json, os, sys, statistics
from collections import Counter, defaultdict
from pathlib import Path
import pandas as pd, yaml
REPO = Path.cwd().parent.parent
sys.path.insert(0, str(REPO)); sys.path.insert(0, str(REPO / "quick_runs"))
RUNS = Path(os.environ.get("REVERIES_RUNS", "/data/anon/reveries_runs"))
TASKS = {t["id"]: t for t in yaml.safe_load(open(REPO / "quick_runs/tasks_p12.yaml"))["tasks"]}
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30); pd.set_option("display.max_colwidth", 80)

In [2]:
from metrics import decision_agreement, first_divergence, answer_consistency, consistent_failures, score_spread, task_score
from reveries.decisions import DECISION_TYPES
from score import family_of, SF_CHECKS
DTYPES = [t for t in DECISION_TYPES if t != "final_selection"] + ["final_selection"]
def load_records(probe):
    recs = [json.loads(l) for l in open(RUNS / probe / "records.jsonl")]
    for r in recs:
        r["answer"] = tuple(r["answer"]) if isinstance(r["answer"], list) else r["answer"]
        r["risk"] = TASKS[r["task_id"]]["contamination_risk"]
        r["source"] = "SF" if r["task_id"].startswith("sf_") else "BLADE"
    return recs
def by_task(recs):
    d = defaultdict(list)
    for r in recs: d[r["task_id"]].append(r)
    return dict(sorted(d.items()))
def task_table(groups):
    rows = []
    for t, rs in groups.items():
        row = {"task": t, "risk": TASKS[t]["contamination_risk"], "n": len(rs),
               "answer_consistency": answer_consistency(rs), "score_mean": task_score(rs), "score_sd": score_spread(rs),
               "first_divergence": first_divergence(rs, DTYPES),
               "consistent_failures": ",".join(consistent_failures(rs))}
        for dt in DTYPES:
            row[dt] = decision_agreement(rs, dt)
        rows.append(row)
    return pd.DataFrame(rows)

In [3]:
import re
TASKS4 = ["sf_tf01_goal_imbalance", "sf_tf02_player_rep_norm", "sf_tf03_correlation_causation", "sf_tf05_credit_card_leakage"]
SF6 = TASKS4 + ["sf_tf04_balance_torque_leakfree", "sf_tf06_metric_select_leakfree"]
TARGET = {"sf_tf01_goal_imbalance": ["checks_class_distribution", "uses_class_weight", "uses_resampling", "uses_stratified_split"],
          "sf_tf02_player_rep_norm": ["handles_unit_suffix_in_height"], "sf_tf03_correlation_causation": ["column_x_included", "applies_causal_reasoning"],
          "sf_tf05_credit_card_leakage": ["used_leakage_features"]}
WRONG_COL = {"sf_tf01_goal_imbalance": "assists", "sf_tf02_player_rep_norm": "height_in_cm",
             "sf_tf03_correlation_causation": "assists", "sf_tf05_credit_card_leakage": "reports"}
FIX = {"sf_tf05_credit_card_leakage": {164, 166, 167, 168, 169}, "sf_tf03_correlation_causation": {127, 129, 131, 132, 134, 135, 137}}
CONTRA3 = {130, 136}; WRONG_IDS = set(range(708, 720))
INSTR = r"task (says|asks|states|requires|instructs)|instruction|use all|all available|contradict"
T14 = [json.loads(l) for l in open(REPO / "quick_runs/results/P14/guide_trace.jsonl")]
T13 = [json.loads(l) for l in open(REPO / "quick_runs/results/P13/guide_trace.jsonl")]
print(Counter(t["arm"] for t in T14))

Counter({'G3o': 30, 'G3n': 30, 'G3o-p': 20, 'G3n-p': 20})


In [4]:
# Stage 1: fix lessons and wrong lessons, per guide arm (runs out of 5 or 20). P13 G3 / G3-p for reference.
def stage1(trs, arm):
    row = {"arm": arm}
    for t, ids in FIX.items():
        rs = [x for x in trs if x["arm"] == arm.split(" ")[0] and x["task"] == t]
        if not rs: continue
        row[t[:7] + " fix seen"] = sum(bool(set(x["seen"]) & ids) for x in rs)
        row[t[:7] + " fix shown"] = sum(bool(set(x["shown"]) & ids) for x in rs)
        row[t[:7] + " n"] = len(rs)
    if t := [x for x in trs if x["task"] == "sf_tf03_correlation_causation" and x["arm"] == arm.split(" ")[0]]:
        row["tf03 contra shown"] = sum(bool(set(x["shown"]) & CONTRA3) for x in t)
    return row
def wrong(trs, arm):
    rs = [x for x in trs if x["arm"] == arm]
    return {"arm": arm, "runs": len(rs), "wrong seen": sum(bool(set(x["seen"]) & WRONG_IDS) for x in rs),
            "wrong accepted": sum(bool(set(x["accepted"]) & WRONG_IDS) for x in rs), "wrong shown": sum(bool(set(x["shown"]) & WRONG_IDS) for x in rs),
            **{f"{t[:7]} shown": sum(bool(set(x["shown"]) & WRONG_IDS) for x in rs if x["task"] == t) for t in TASKS4}}
s1 = pd.DataFrame([stage1(T13, "G3"), stage1(T14, "G3o"), stage1(T14, "G3n")]).set_index("arm")
wr = pd.DataFrame([wrong(T13, "G3-p"), wrong(T14, "G3o-p"), wrong(T14, "G3n-p")]).set_index("arm")
print(s1.to_string()); wr

     sf_tf05 fix seen  sf_tf05 fix shown  sf_tf05 n  sf_tf03 fix seen  sf_tf03 fix shown  sf_tf03 n  tf03 contra shown
arm                                                                                                                   
G3                  5                  0          5                 5                  1          5                  4
G3o                 5                  0          5                 5                  1          5                  4
G3n                 5                  2          5                 5                  0          5                  5


,runs,wrong seen,wrong accepted,wrong shown,sf_tf01 shown,sf_tf02 shown,sf_tf03 shown,sf_tf05 shown
arm,,,,,,,,
G3-p,20,20,5,5,4,0,0,1
G3o-p,20,20,6,6,5,0,0,1
G3n-p,20,20,10,9,5,0,0,4


In [5]:
import sys, yaml
sys.path.insert(0, str(REPO / "quick_runs" / "p13")); sys.path.insert(0, str(REPO / "quick_runs" / "p12"))
import guide as GD
from scipy.stats import fisher_exact
fp = lambda a, na, b, nb: round(fisher_exact([[a, na - a], [b, nb - b]])[1], 4)
def labids(store, labfile, label):
    LAB = yaml.safe_load(open(REPO / labfile))
    return {i for i, L in enumerate(store) if L.get("nf") and L["source_task"] in LAB
            and f"{L['source_run']}/{L['k']}" in (LAB[L["source_task"]][label] or [])}
NFDTR = GD.load_store("nfdtr")
FIXD = labids(NFDTR, "quick_runs/p21/labels_nfdtr.yaml", "fix")
REINFD = labids(NFDTR, "quick_runs/p21/labels_nfdtr.yaml", "reinforcing")
nfd_n = Counter(L["source_task"][:7] for L in NFDTR if L.get("probe") == "train")
labd = {t[:7]: {k: len(v or []) for k, v in d.items()} for t, d in yaml.safe_load(open(REPO / "quick_runs/p21/labels_nfdtr.yaml")).items()}
print("nfdtr lessons:", sum(nfd_n.values()), dict(nfd_n)); print("labels:", labd)
# exposure: which fix / reinforcing lessons the C1-nfdtr and E-nfdtr guide stages showed
TN = [json.loads(l) for l in open(REPO / "quick_runs/results/P21/trace_nfdtr.jsonl")]
expo = pd.DataFrame([{"arm": t["arm"], "task": t["task"][:7], "fix shown": bool(set(t["shown"]) & FIXD),
                      "reinf shown": bool(set(t["shown"]) & REINFD)} for t in TN if t["task"] in TASKS4]).groupby(["arm", "task"]).sum()
expo

nfdtr lessons: 115 {'sf_tf01': 15, 'sf_tf02': 15, 'sf_tf03': 13, 'sf_tf04': 28, 'sf_tf05': 14, 'sf_tf06': 30}
labels: {'sf_tf01': {'fix': 3, 'reinforcing': 0}, 'sf_tf02': {'fix': 1, 'reinforcing': 4}, 'sf_tf03': {'fix': 5, 'reinforcing': 2}, 'sf_tf05': {'fix': 0, 'reinforcing': 0}}


fix shown  reinf shown
arm      task                           
C1-nfdtr sf_tf01          5            0
         sf_tf02          5            5
         sf_tf03          5            5
         sf_tf05          0            0
E-nfdtr  sf_tf01          5            0
         sf_tf02          0            0
         sf_tf03          0            5
         sf_tf05          0            0

In [6]:
# Stage 2: agent runs. Targeted checks on the 4 failing tasks (passes out of 5), P14 G3n next to P12/P13 arms
def load(arm):
    recs = [json.loads(l) for l in open(RUNS / arm / "records.jsonl")]
    recs = [r for r in recs if r["task_id"] in SF6]
    for r in recs:
        ev = [json.loads(l) for l in open(RUNS / arm / r["task_id"] / r["run"] / "events.jsonl")]
        calls = [e for e in ev if e["event"] == "llm_call" and e.get("usage")]
        r["sum_tokens"] = sum(c["usage"]["prompt_tokens"] + c["usage"]["completion_tokens"] for c in calls)
    return recs
ARMS = ["P12_A0", "P12_C1", "P21_A0g2", "P21_C1g2", "P13_E", "P17_C1nf", "P19_C1nfd", "P21_C1nfdtr", "P21_Enfdtr"]
NAME = dict(zip(ARMS, ["A0 4gpu", "C1 4gpu", "A0 2gpu", "C1 2gpu", "E emb", "C1-nf", "C1-nfd", "C1-nfdtr", "E-nfdtr"]))
recs = {a: load(a) for a in ARMS}
print({a: (len(v), dict(Counter(r["stop"] for r in v))) for a, v in recs.items()})
rows = []
for t, checks in TARGET.items():
    for c in checks:
        rows.append({"task": t[:7], "check": c, **{NAME[a]: sum(r["checks"].get(c) is True for r in recs[a] if r["task_id"] == t) for a in ARMS}})
targeted = pd.DataFrame(rows)
pooled = targeted[[NAME[a] for a in ARMS]].sum() / 40
print("pooled:", pooled.round(3).to_dict()); targeted

{'P12_A0': (30, {'final_answer': 30}), 'P12_C1': (30, {'final_answer': 29, 'max_steps': 1}), 'P21_A0g2': (30, {'final_answer': 30}), 'P21_C1g2': (30, {'final_answer': 28, 'max_steps': 2}), 'P13_E': (30, {'final_answer': 29, 'max_steps': 1}), 'P17_C1nf': (30, {'final_answer': 30}), 'P19_C1nfd': (30, {'final_answer': 30}), 'P21_C1nfdtr': (30, {'final_answer': 30}), 'P21_Enfdtr': (30, {'final_answer': 30})}
pooled: {'A0 4gpu': 0.075, 'C1 4gpu': 0.75, 'A0 2gpu': 0.075, 'C1 2gpu': 0.65, 'E emb': 0.475, 'C1-nf': 0.025, 'C1-nfd': 0.6, 'C1-nfdtr': 0.425, 'E-nfdtr': 0.275}


,task,check,A0 4gpu,C1 4gpu,A0 2gpu,C1 2gpu,E emb,C1-nf,C1-nfd,C1-nfdtr,E-nfdtr
0,sf_tf01,checks_class_distribution,0,5,1,5,5,0,4,3,3
1,sf_tf01,uses_class_weight,0,5,0,5,5,0,5,5,5
2,sf_tf01,uses_resampling,0,0,0,0,0,0,0,0,0
3,sf_tf01,uses_stratified_split,3,5,2,5,5,1,5,1,3
4,sf_tf02,handles_unit_suffix_in_height,0,4,0,3,4,0,0,0,0
5,sf_tf03,column_x_included,0,5,0,4,0,0,5,5,0
6,sf_tf03,applies_causal_reasoning,0,5,0,4,0,0,5,3,0
7,sf_tf05,used_leakage_features,0,1,0,0,0,0,0,0,0


In [7]:
# Hardware split: per-task targeted checks, 4-GPU (P12) vs 2-GPU (P21) for A0 and C1
hw = targeted.set_index(["task", "check"])[["A0 4gpu", "A0 2gpu", "C1 4gpu", "C1 2gpu"]]
hw["A0 diff"] = (hw["A0 4gpu"] - hw["A0 2gpu"]).abs(); hw["C1 diff"] = (hw["C1 4gpu"] - hw["C1 2gpu"]).abs()
print("max per-check diff: A0", int(hw["A0 diff"].max()), "C1", int(hw["C1 diff"].max()))
hw

max per-check diff: A0 1 C1 1


A0 4gpu  A0 2gpu  C1 4gpu  C1 2gpu  A0 diff  C1 diff
task    check                                                                              
sf_tf01 checks_class_distribution            0        1        5        5        1        0
        uses_class_weight                    0        0        5        5        0        0
        uses_resampling                      0        0        0        0        0        0
        uses_stratified_split                3        2        5        5        1        0
sf_tf02 handles_unit_suffix_in_height        0        0        4        3        0        1
sf_tf03 column_x_included                    0        0        5        4        0        1
        applies_causal_reasoning             0        0        5        4        0        1
sf_tf05 used_leakage_features                0        0        1        0        0        1

In [8]:
# Harm and exact tests
sc = pd.DataFrame([{"arm": NAME[a], "task": r["task_id"], "score": r["score"]} for a in ARMS for r in recs[a]])
scores = sc.pivot_table(index="task", columns="arm", values="score", aggfunc="mean").round(4)
passed = (pooled * 40).round().astype(int)
fisher = {"C1-nf vs C1-nfdtr": fp(passed["C1-nf"], 40, passed["C1-nfdtr"], 40),
          "C1-nfd vs C1-nfdtr": fp(passed["C1-nfd"], 40, passed["C1-nfdtr"], 40),
          "C1-nfdtr vs C1 4gpu": fp(passed["C1-nfdtr"], 40, passed["C1 4gpu"], 40),
          "A0 4gpu vs A0 2gpu": fp(passed["A0 4gpu"], 40, passed["A0 2gpu"], 40),
          "C1 4gpu vs C1 2gpu": fp(passed["C1 4gpu"], 40, passed["C1 2gpu"], 40)}
print(fisher); scores

{'C1-nf vs C1-nfdtr': np.float64(0.0), 'C1-nfd vs C1-nfdtr': np.float64(0.1793), 'C1-nfdtr vs C1 4gpu': np.float64(0.006), 'A0 4gpu vs A0 2gpu': 1.0, 'C1 4gpu vs C1 2gpu': np.float64(0.4647)}


arm,A0 2gpu,A0 4gpu,C1 2gpu,C1 4gpu,C1-nf,C1-nfd,C1-nfdtr,E emb,E-nfdtr
task,,,,,,,,,
sf_tf01_goal_imbalance,7.049000e-01,7.603000e-01,7.868000e-01,7.765000e-01,7.860000e-01,7.715000e-01,7.704000e-01,7.713000e-01,7.454000e-01
sf_tf02_player_rep_norm,4.664201e+06,4.664201e+06,4.650909e+06,4.650909e+06,4.664201e+06,4.664201e+06,4.664201e+06,4.650909e+06,4.664201e+06
sf_tf03_correlation_causation,2.769462e+07,2.769462e+07,9.337854e+06,4.755266e+06,2.769458e+07,4.699080e+06,4.748095e+06,2.769464e+07,2.769459e+07
sf_tf04_balance_torque_leakfree,9.149000e-01,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00
sf_tf05_credit_card_leakage,9.659000e-01,9.659000e-01,8.220000e-01,8.220000e-01,9.659000e-01,9.659000e-01,9.659000e-01,9.144000e-01,9.659000e-01
sf_tf06_metric_select_leakfree,9.789000e-01,9.798000e-01,9.806000e-01,9.806000e-01,9.797000e-01,9.799000e-01,9.797000e-01,9.796000e-01,9.777000e-01


In [9]:
summary = {"nfdtr_lessons": dict(nfd_n), "nfdtr_labels": labd,
  "nfdtr_exposure": expo.reset_index().to_dict("records"),
  "runs": {a: len(v) for a, v in recs.items()}, "stops": {a: dict(Counter(r["stop"] for r in v)) for a, v in recs.items()},
  "targeted": targeted.to_dict("records"), "targeted_pooled": pooled.round(3).to_dict(),
  "hardware": hw.reset_index().to_dict("records"), "fisher_p": fisher, "score_mean": scores.to_dict("index")}
json.dump(summary, open(REPO / "quick_runs/results/P21/summary.json", "w"), indent=1, default=str)
"done" 

'done'